# Part_5_Exercice_1_wikipedia.ipynb

In [2]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

## Étape 1 : Charger le HTML depuis le fichier local

**Option 1**: Au lieu de faire une requête HTTP, on lit directement le fichier html.md

In [3]:
with open("celadon_html.md", "r", encoding="utf-8") as f:
    html_content = f.read()

# Parser le HTML avec BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

**Option 2** : Utiliser l'url

In [4]:
headers = {
    'User-Agent': 'MonBotApprentissage/1.0 (formation scraping; contact@example.com)',
    'Accept-Language': 'fr-FR,fr;q=0.9',
    'Accept': 'text/html,application/xhtml+xml',
    'From': 'contact@example.com'  # Email de contact (bonne pratique)
}


url = "https://celadon-paris.com/collections/tasses-et-mugs-vaisselle-ceramique-gres-portugal-the-cafe?srsltid=AfmBOop1_8UuMUTEhoNSUbmhyIpIR3LZu96DEOAsOgpAYXKgTCEVasoP"

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
print(soup)

<!DOCTYPE html>

<html class="no-js" lang="fr">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1.0, height=device-height, minimum-scale=1.0, maximum-scale=1.0" name="viewport"/>
<meta content="" name="theme-color"/>
<title>
      Tasses &amp; Mugs en grès - Vaisselle en céramique du Portugal
      
      
       – Céladon Paris
    </title><meta content="Thé fumant, café du matin ou chocolat chaud du goûter : il y a un mug pour chaque moment. Déclinés en trois formats, nos mugs en grès sont façonnés à la main au Portugal. Solides, durables et émaillés dans nos couleurs signature, ils apportent une touche artisanale et chaleureuse à toutes vos pauses." name="description"/><link href="https://celadon-paris.com/collections/tasses-et-mugs-vaisselle-ceramique-gres-portugal-the-cafe" rel="canonical"/><link href="//celadon-paris.com/cdn/shop/files/logo_celadon_fav_a9f211e8-e53d-4431-9232-a55b034cf

## Étape 2 : Cibler le conteneur principal
On cherche le conteneur principal qui contient tous les produits

In [5]:
conteneur_produits = soup.select_one("div.ProductList")
print(f"Conteneur trouvé : {conteneur_produits is not None}")

Conteneur trouvé : True


## Étape 3 : Trouver tous les produits 
Chaque produit est dans une div avec la classe "ProductItem"

In [6]:
tous_les_produits = soup.select("div.ProductItem")
print(f"Nombre de produits trouvés : {len(tous_les_produits)}")

Nombre de produits trouvés : 28


In [8]:
print(tous_les_produits)

[<div class="ProductItem">
<div class="ProductItem__Wrapper"><a class="ProductItem__ImageWrapper ProductItem__ImageWrapper--withAlternateImage" href="/products/grand-mug-leopard-vaisselle-ceramique-imprime"><div class="AspectRatio AspectRatio--withFallback" style="max-width: 1000px; padding-bottom: 100.0%; --aspect-ratio: 1.0"><img alt="Grand Mug | Léopard" class="ProductItem__Image ProductItem__Image--alternate Image--lazyLoad Image--fadeIn" data-media-id="54586873643335" data-sizes="auto" data-src="//celadon-paris.com/cdn/shop/files/Mug-leopard-8_{width}x.jpg?v=1764696078" data-widths="[200,300,400,600,800,900,1000]"/><img alt="Grand Mug | Léopard" class="ProductItem__Image Image--lazyLoad Image--fadeIn" data-media-id="55926906519879" data-sizes="auto" data-src="//celadon-paris.com/cdn/shop/files/18_5d6a2bda-8155-4ec3-8903-1f5b3a31f937_{width}x.jpg?v=1764696078" data-widths="[200,400,600,700,800,900,1000]"/>
<span class="Image__Loader"></span>
<noscript>
<img alt="Grand Mug | Léopard

## Étape 4 : Extraire les données de chaque produit
On définit les colonnes qu'on veut extraire

In [7]:
noms_des_colonnes = ["Nom", "Prix", "Note", "Nombre d'avis", "Lien", "Label"]

# Liste pour stocker les données extraites
donnees = []

for produit in tous_les_produits:
    # Extraire le nom du produit
    titre_element = produit.select_one("h2.ProductItem__Title a")
    nom = titre_element.text.strip() if titre_element else "N/A"

    # Extraire le lien du produit
    lien = titre_element.get("href", "N/A") if titre_element else "N/A"

    # Extraire le prix
    prix_element = produit.select_one("span.ProductItem__Price")
    prix = prix_element.text.strip() if prix_element else "N/A"

    # Extraire la note (dans l'attribut aria-label)
    rating_element = produit.select_one("div.rating__stars")
    if rating_element and rating_element.has_attr("aria-label"):
        # Format: "4.82 sur 5.0 étoiles"
        aria_label = rating_element.get("aria-label", "")
        note = aria_label.split(" ")[0] if aria_label else "N/A"
    else:
        note = "N/A"

    # Extraire le nombre d'avis
    avis_element = produit.select_one("span.rating__caption")
    if avis_element:
        # Format: "34 avis"
        nb_avis = avis_element.text.strip().replace(" avis", "")
    else:
        nb_avis = "N/A"

    # Extraire le label (ex: "ÉDITION LIMITÉE")
    label_element = produit.select_one("span.ProductItem__Label")
    label = label_element.text.strip() if label_element else ""

    # Ajouter les données à notre liste
    donnees.append((nom, prix, note, nb_avis, lien, label))

    # Afficher les données extraites (comme dans le notebook Wikipedia)
    print(f"{nom} | {prix} | Note: {note}/5 | {nb_avis} avis | {label}")

Grand Mug | Léopard | 29,00€ | Note: 4.82/5 | 34 avis | ÉDITION LIMITÉE
Grand Mug | Noir écume | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Vert rivage | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Terracotta sienna | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Bleu água | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Blanc audacieux | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Bleu ciel | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Vert audacieux | 20,00€ | Note: 4.96/5 | 605 avis | 
Grand Mug | Bleu odyssée | 20,00€ | Note: 4.96/5 | 605 avis | BIENTÔT DE RETOUR
Grand Mug | Rose Bohême | 20,00€ | Note: 4.96/5 | 605 avis | 
Mug moyen | Noir écume | 18,00€ | Note: 4.97/5 | 258 avis | 
Mug moyen | Bleu ciel | 18,00€ | Note: 4.97/5 | 258 avis | 
Mug moyen | Bleu odyssée | 18,00€ | Note: 4.97/5 | 258 avis | 
Mug moyen | Blanc audacieux | 18,00€ | Note: 4.97/5 | 258 avis | 
Mug moyen | Terracotta sienna | 18,00€ | Note: 4.97/5 | 258 avis | 
Mug moyen | Vert rivage | 

## Étape 5 : Créer un DataFrame pandas pour visualiser les données

In [8]:
df_produits = pd.DataFrame(donnees, columns=noms_des_colonnes)

print("="*80)
print("APERÇU DU DATAFRAME")
print("="*80)
df_produits.head(10)

APERÇU DU DATAFRAME


,Nom,Prix,Note,Nombre d'avis,Lien,Label
0,Grand Mug | Léopard,"29,00€",4.82,34,/products/grand-mug-leopard-vaisselle-ceramiqu...,ÉDITION LIMITÉE
1,Grand Mug | Noir écume,"20,00€",4.96,605,/products/grand-mug-noir-ecume-vaisselle-ceram...,
2,Grand Mug | Vert rivage,"20,00€",4.96,605,/products/grand-mug-vert-rivage-vaisselle-tass...,
3,Grand Mug | Terracotta sienna,"20,00€",4.96,605,/products/grand-mug-terracotta-sienna-vaissell...,
4,Grand Mug | Bleu água,"20,00€",4.96,605,/products/grand-mug-bleu-agua-tasse-ceramique-...,
5,Grand Mug | Blanc audacieux,"20,00€",4.96,605,/products/grand-mug-blanc-audacieux-ceramique-...,
6,Grand Mug | Bleu ciel,"20,00€",4.96,605,/products/grand-mug-rond-bleu-ciel-vaisselle-c...,
7,Grand Mug | Vert audacieux,"20,00€",4.96,605,/products/grand-mug-vert-audacieux-ceramique-v...,
8,Grand Mug | Bleu odyssée,"20,00€",4.96,605,/products/grand-mug-bleu-odyssee-ceramique-vai...,BIENTÔT DE RETOUR
9,Grand Mug | Rose Bohême,"20,00€",4.96,605,/products/grand-mug-rose-boheme-tasse-ceramiqu...,


In [ ]:
print("="*80)
print("STATISTIQUES")
print("="*80)
print(f"Nombre total de produits : {len(df_produits)}")
print(f"\nDistribution des prix :")
df_produits["Prix"].value_counts().head(10)

STATISTIQUES
Nombre total de produits : 28

Distribution des prix :


Prix
20,00€    9
18,00€    9
16,00€    9
29,00€    1
Name: count, dtype: int64

## Étape 6 : Exporter les données en CSV (optionnel)

In [ ]:
df_produits.to_csv("produits_celadon.csv", index=False, encoding="utf-8")
print("Données exportées dans 'produits_celadon.csv'")